# Doubly Nested i2Nav LOSO Leakage-Repair Study

This notebook replaces the older ordinary 10-fold x 3-seed notebook for the **two-stage provenance repair**. For each outer qualification-test sequence `u`, it creates: (1) an outer-test trajectory from a checkpoint excluding `u`; and (2) each qualification-training trajectory `v != u` from a checkpoint excluding both `u` and `v`. Training, normalization, validation, and checkpoint selection therefore exclude `u` for every trajectory used in outer fold `u`.

The full design is 10 outer folds x (1 outer + 9 qualification-training trajectories) x 3 seeds = **300 paired V1-to-V2 pipelines (600 model fits)**. Both the V1 covariance model and V2 correction model obey the exclusions. Run the design in deterministic shards across Kaggle sessions. A shard is not a complete scientific result. Do not compare headline counts until every required shard has been merged and audited.

In [ ]:

from pathlib import Path
import csv, hashlib, json, os, shutil, subprocess, sys, time, zipfile

REPO_URL = 'https://github.com/CEISCA-VT/DigitalTwinDivergence.git'
REPO_REF = 'main'
EXPECTED_COMMIT = '6f91f5dae834ac55dfebf3abae76beba7379ff2c'
STUDY_MODE = 'full10'  # 'full10' or 'subset5'
SUBSET5 = ['parking01', 'parking02', 'building00', 'playground00', 'street00']
BASE_SEEDS = [42, 1042, 2042]
SHARD_INDEX = 0       # zero based
SHARD_COUNT = 1     # full10: 5 pipelines / 10 model fits per session
DEVICE = 'cuda'

WORK = Path('/kaggle/working')
REPO = WORK / 'DigitalTwinDivergence'
OUTPUT = WORK / 'i2nav_doubly_nested_loso'
LOGS = WORK / 'i2nav_doubly_nested_logs'
OUTPUT.mkdir(parents=True, exist_ok=True); LOGS.mkdir(parents=True, exist_ok=True)
if not EXPECTED_COMMIT or EXPECTED_COMMIT == 'REPLACE_WITH_PUSHED_COMMIT' or len(EXPECTED_COMMIT) != 40: raise ValueError('Set EXPECTED_COMMIT to the full pushed repository commit before launching shards')
if not (0 <= SHARD_INDEX < SHARD_COUNT): raise ValueError('Invalid shard configuration')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',REPO_REF,REPO_URL,str(REPO)], check=True)
COMMIT = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
if COMMIT != EXPECTED_COMMIT: raise RuntimeError(f'Commit mismatch: {COMMIT} != {EXPECTED_COMMIT}')
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'

def sha256_file(path, block_size=1024*1024):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(block_size), b''):
            h.update(block)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text())

print({'commit': COMMIT, 'study_mode': STUDY_MODE, 'shard': [SHARD_INDEX, SHARD_COUNT]})


In [ ]:
# GPU gate. Torch stays in child processes so a compatibility reinstall is safe.
probe = "import json,torch; print(json.dumps({'ok':torch.cuda.is_available(),'cap':list(torch.cuda.get_device_capability(0)) if torch.cuda.is_available() else None,'arch':torch.cuda.get_arch_list() if torch.cuda.is_available() else []}))"
p = subprocess.run([sys.executable,'-c',probe], text=True, capture_output=True, check=True)
gpu = json.loads(p.stdout.strip().splitlines()[-1]); print(gpu)
if not gpu['ok']: raise RuntimeError('Enable a Kaggle GPU and restart the session')
if tuple(gpu['cap']) == (6,0) and 'sm_60' not in gpu['arch']:
    subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','--force-reinstall','torch==2.5.1','--index-url','https://download.pytorch.org/whl/cu118'], check=True)
real = "import torch; d='cuda'; x=torch.randn(128,128,device=d); g=torch.nn.GRU(6,64,2,batch_first=True).to(d); y,_=g(torch.randn(2,20,6,device=d)); torch.cuda.synchronize(); assert torch.isfinite(x@x.T).all() and torch.isfinite(y).all(); print(torch.__version__,torch.cuda.get_device_name(0))"
subprocess.run([sys.executable,'-c',real], check=True)
print('CUDA PREFLIGHT: PASS')


In [ ]:
# Build the complete deterministic task ledger before running anything.
v1_runner='DigitalTwin.analysis.i2nav_loso_ablation'; v2_runner='DigitalTwin.analysis.i2nav_v2_full_loso'
v1_help=subprocess.check_output([sys.executable,'-m',v1_runner,'--help'],cwd=REPO,text=True)
v2_help=subprocess.check_output([sys.executable,'-m',v2_runner,'--help'],cwd=REPO,text=True)
if '--additional-excluded-sequence' not in v1_help or '--v1-checkpoint' not in v2_help: raise RuntimeError('Repository commit lacks full nested V1/V2 support')
seqs = ['building00','building01','building02','parking00','parking01','parking02','playground00','street00','street01','street02']
outer = seqs if STUDY_MODE == 'full10' else SUBSET5
if STUDY_MODE not in {'full10','subset5'}: raise ValueError(STUDY_MODE)
for name in seqs:
    if not (REPO/'public_datasets'/'im2nav'/name).is_dir(): raise FileNotFoundError(name)
if not (REPO/'results'/'i2nav_v1_frozen'/'FROZEN_MANIFEST.json').is_file(): raise FileNotFoundError('Frozen V1 evidence')
tasks=[]
for u in outer:
    for seed in BASE_SEEDS:
        tasks.append({'outer':u,'target':u,'seed':seed,'role':'outer_test','extra_excluded':None})
    for v in seqs:
        if v == u: continue
        for seed in BASE_SEEDS:
            tasks.append({'outer':u,'target':v,'seed':seed,'role':'qualification_train','extra_excluded':u})
tasks = sorted(tasks, key=lambda x:(x['outer'],x['role'],x['target'],x['seed']))
shard_tasks=[t for i,t in enumerate(tasks) if i % SHARD_COUNT == SHARD_INDEX]
ledger={'schema':'doubly_nested_loso_task_ledger_v1','commit':COMMIT,'study_mode':STUDY_MODE,'outer_sequences':outer,'base_seeds':BASE_SEEDS,'total_tasks':len(tasks),'shard_index':SHARD_INDEX,'shard_count':SHARD_COUNT,'shard_tasks':shard_tasks}
(OUTPUT/f'task_ledger_shard_{SHARD_INDEX:02d}.json').write_text(json.dumps(ledger,indent=2)+'\n')
print(f'Total={len(tasks)}; this shard={len(shard_tasks)}')
print(json.dumps(shard_tasks[:3],indent=2))


In [ ]:

# Resumable execution: fresh V1 and V2 processes per pipeline. Never ensemble predictions.
def v1_complete(v1_out, target):
    checkpoint = v1_out / 'folds' / target / 'gru_dual.pt'
    results = v1_out / 'loso_results.csv'
    split = v1_out / 'fold_splits.json'
    if not (checkpoint.is_file() and results.is_file() and split.is_file()):
        return False
    rows = read_json(split)
    if len(rows) != 1 or rows[0].get('test') != target:
        return False
    with results.open(newline='', encoding='utf-8') as f:
        return any(r.get('method') == 'gru_dual' and r.get('status') == 'ok' for r in csv.DictReader(f))

def v2_manifest_for(parent, target, seed):
    found = []
    for p in (parent / 'v2').rglob(f'*_{target}/run_manifest.json'):
        try:
            m = read_json(p)
        except Exception:
            continue
        if m.get('base_seed') == seed:
            found.append((p, m))
    return found

def v2_complete(parent, target, seed, expected_v1, expected_results):
    matches = v2_manifest_for(parent, target, seed)
    if len(matches) != 1:
        return False
    p, m = matches[0]
    marker = p.with_name('RUN_COMPLETE.json')
    required = [
        marker,
        p.with_name('run_summary.json'),
        p.with_name('v2_slow_additive_yaw.pt'),
        p.with_name('v2_evaluated_trajectory.csv'),
        p.with_name('v2_prediction_trace.csv'),
        p.with_name('fidelity_profile.json'),
        p.with_name('fidelity_timeseries.csv'),
    ]
    if not all(x.is_file() for x in required):
        return False
    return (
        m.get('status') == 'complete'
        and Path(m.get('v1_checkpoint_source','')).resolve() == expected_v1.resolve()
        and Path(m.get('v1_results_source','')).resolve() == expected_results.resolve()
    )

env=os.environ.copy(); env['PYTHONPATH']=str(REPO)
for index,t in enumerate(shard_tasks,1):
    parent=OUTPUT/f"outer_{t['outer']}"/t['role']/f"target_{t['target']}"; v1_out=parent/f"v1_seed_{t['seed']}"; v2_out=parent/'v2'
    v1_cmd=[sys.executable,'-m',v1_runner,'--root',str(REPO/'public_datasets'/'im2nav'),'--output-dir',str(v1_out),'--folds',t['target'],'--methods','gru_dual','--save-trajectories','--seed',str(t['seed']),'--device',DEVICE]
    if t['extra_excluded']: v1_cmd += ['--additional-excluded-sequence',t['extra_excluded']]
    v1_checkpoint=v1_out/'folds'/t['target']/'gru_dual.pt'; v1_results=v1_out/'loso_results.csv'
    v2_cmd=[sys.executable,'-m',v2_runner,'--root',str(REPO/'public_datasets'/'im2nav'),'--frozen-v1-dir',str(REPO/'results'/'i2nav_v1_frozen'),'--output-dir',str(v2_out),'--test-sequence',t['target'],'--base-seed',str(t['seed']),'--device',DEVICE,'--v1-checkpoint',str(v1_checkpoint),'--v1-results-csv',str(v1_results)]
    if t['extra_excluded']: v2_cmd += ['--additional-excluded-sequence',t['extra_excluded']]
    log=LOGS/f"outer-{t['outer']}__role-{t['role']}__target-{t['target']}__seed-{t['seed']}.log"
    print(f"[{index}/{len(shard_tasks)}] {t}",flush=True)
    if v1_complete(v1_out, t['target']):
        print('  V1 complete; skipping')
    else:
        with log.open('a',encoding='utf-8') as f:
            result=subprocess.run(v1_cmd,cwd=REPO,env=env,text=True,stdout=f,stderr=subprocess.STDOUT)
        if result.returncode: raise RuntimeError(f'V1 fit failed; inspect {log}')
    if v2_complete(parent, t['target'], t['seed'], v1_checkpoint, v1_results):
        print('  V2 complete; skipping')
        continue
    with log.open('a',encoding='utf-8') as f:
        result=subprocess.run(v2_cmd,cwd=REPO,env=env,text=True,stdout=f,stderr=subprocess.STDOUT)
    if result.returncode: raise RuntimeError(f'V2 fit failed; inspect {log}')
print('SHARD EXECUTION COMPLETE')


In [ ]:

# Audit only this shard. Scientific aggregation requires all shards.
audit=[]
for t in shard_tasks:
    parent=OUTPUT/f"outer_{t['outer']}"/t['role']/f"target_{t['target']}"; v1_out=parent/f"v1_seed_{t['seed']}"
    v1_checkpoint=(v1_out/'folds'/t['target']/'gru_dual.pt').resolve(); v1_results=(v1_out/'loso_results.csv').resolve()
    split=read_json(v1_out/'fold_splits.json')[0]; v1_used=set(split['train'])|set(split['validation'])
    if t['target'] in v1_used or (t['extra_excluded'] and t['extra_excluded'] in v1_used): raise RuntimeError(f'V1 split leakage: {t}')
    for key in ['normalization_fit_sequences','checkpoint_selection_sequences']:
        if key not in split: raise RuntimeError(f'Missing V1 provenance field {key}: {t}')
    if set(split['normalization_fit_sequences']) != set(split['train']): raise RuntimeError(f'V1 normalization provenance mismatch: {t}')
    if set(split['checkpoint_selection_sequences']) != set(split['validation']): raise RuntimeError(f'V1 checkpoint-selection provenance mismatch: {t}')
    matches=v2_manifest_for(parent, t['target'], t['seed'])
    if len(matches)!=1: raise RuntimeError(f'Manifest count for {t}: {len(matches)}')
    p,m=matches[0]; used=set(m['training_names'])|set(m['validation_names'])
    if m.get('repo_commit') != COMMIT: raise RuntimeError(f'V2 commit mismatch: {p}')
    if t['target'] in used: raise RuntimeError(f"Target leakage: {t}")
    if t['extra_excluded'] and t['extra_excluded'] in used: raise RuntimeError(f"Outer leakage: {t}")
    if set(m.get('normalization_fit_sequences', [])) != set(m['training_names']): raise RuntimeError(f'V2 normalization provenance mismatch: {p}')
    if set(m.get('checkpoint_selection_sequences', [])) != set(m['validation_names']): raise RuntimeError(f'V2 checkpoint-selection provenance mismatch: {p}')
    if not p.with_name('RUN_COMPLETE.json').is_file(): raise RuntimeError(f'Incomplete: {p}')
    if Path(m.get('v1_checkpoint_source','')).resolve()!=v1_checkpoint: raise RuntimeError(f'V2 did not use exact restricted V1 checkpoint: {t}')
    if Path(m.get('v1_results_source','')).resolve()!=v1_results: raise RuntimeError(f'V2 did not use exact restricted V1 results: {t}')
    if m.get('v1_checkpoint_sha256') != sha256_file(v1_checkpoint): raise RuntimeError(f'V1 checkpoint hash mismatch: {t}')
    if m.get('v1_results_sha256') != sha256_file(v1_results): raise RuntimeError(f'V1 results hash mismatch: {t}')
    artifacts=m.get('artifacts_sha256',{})
    traj=p.with_name('v2_evaluated_trajectory.csv')
    if artifacts.get('evaluated_trajectory') != sha256_file(traj): raise RuntimeError(f'V2 trajectory hash mismatch: {t}')
    audit.append({**t,'v1_split':str((v1_out/'fold_splits.json').relative_to(OUTPUT)),'v1_checkpoint_sha256':m.get('v1_checkpoint_sha256'),'v1_results_sha256':m.get('v1_results_sha256'),'v2_trajectory_sha256':artifacts.get('evaluated_trajectory'),'v2_manifest':str(p.relative_to(OUTPUT)),'training':m['training_names'],'validation':m['validation_names'],'normalization_fit_sequences':m['normalization_fit_sequences'],'checkpoint_selection_sequences':m['checkpoint_selection_sequences'],'status':'PASS'})
audit_doc={'schema':'doubly_nested_loso_shard_audit_v2','commit':COMMIT,'shard_index':SHARD_INDEX,'shard_count':SHARD_COUNT,'rows':audit,'complete':len(audit)==len(shard_tasks)}
(OUTPUT/f'shard_audit_{SHARD_INDEX:02d}.json').write_text(json.dumps(audit_doc,indent=2)+'\n')
print(f'AUDIT PASS: {len(audit)}/{len(shard_tasks)} tasks')


In [ ]:

# Package only this shard. Upload every shard archive before merging in the repository.
archive=WORK/f"i2nav_doubly_nested_{STUDY_MODE}_shard_{SHARD_INDEX:02d}_of_{SHARD_COUNT:02d}.zip"
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as z:
    shard_files = [OUTPUT/f'task_ledger_shard_{SHARD_INDEX:02d}.json', OUTPUT/f'shard_audit_{SHARD_INDEX:02d}.json']
    for t in shard_tasks:
        parent=OUTPUT/f"outer_{t['outer']}"/t['role']/f"target_{t['target']}"
        for p in parent.rglob('*'):
            if p.is_file(): shard_files.append(p)
        log=LOGS/f"outer-{t['outer']}__role-{t['role']}__target-{t['target']}__seed-{t['seed']}.log"
        if log.is_file(): shard_files.append(log)
    for p in sorted(set(shard_files)):
        if p.is_file(): z.write(p,p.relative_to(WORK))
with zipfile.ZipFile(archive,'r') as z:
    bad=z.testzip()
    if bad: raise RuntimeError(f'ZIP integrity check failed at {bad}')
print({'archive':str(archive),'sha256':sha256_file(archive),'bytes':archive.stat().st_size})


## Interpretation boundary
A successful shard proves only that its assigned fits completed with the declared exclusions. Merge and audit every shard before recomputing `5/235 versus 75/200`, `30/6/4`, or `27/30`. Keep the original frozen study and this repaired study in separate result roots.